In [1]:
data_file_path="../finance_artifact/data_ingestion/feature_store/finance_complaint"

In [3]:
from finance_complaint.config.spark_manager import spark_session

In [4]:
df = spark_session.read.parquet(data_file_path)

In [5]:
df.count()

6907890

In [6]:
from pyspark.sql.functions import col

In [7]:
df.select('complaint_id')

DataFrame[complaint_id: string]

## Conf

In [8]:
def update_column_attribute(df):
    for column in df.columns:
        setattr(df,column,column)

In [9]:
update_column_attribute(df)

## Printing unique values in each column

In [10]:
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

company:6608


company_public_response:12


company_response:7


complaint_id:3837805


complaint_what_happened:1218265


...

state:64


sub_issue:268


sub_product:87
submitted_via:7


tags:4
timely:2


zip_code:29520


In [11]:
complaint_table="complaint"
df.createOrReplaceTempView(complaint_table)

In [12]:
sql = spark_session.sql

In [13]:
n_row = df.count()

In [14]:
#Target column
df.groupBy(df.consumer_disputed).count().collect()

[Row(consumer_disputed='N/A', count=6465744),
 Row(consumer_disputed='No', count=368588),
 Row(consumer_disputed='Yes', count=73558)]

In [15]:
df.printSchema()

root
 |-- company: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- complaint_what_happened: string (nullable = true)
 |-- consumer_consent_provided: string (nullable = true)
 |-- consumer_disputed: string (nullable = true)
 |-- date_received: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- product: string (nullable = true)
 |-- state: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- timely: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [16]:
missing_target_df = sql(f"select * from {complaint_table} where {df.consumer_disputed} ='N/A' ")

In [17]:
df = sql(f"select * from {complaint_table} where {df.consumer_disputed} <>'N/A' ")

In [18]:
complaint_table="complaint_temp"
df.createOrReplaceTempView(complaint_table)

In [19]:
update_column_attribute(df)

In [20]:
duplicate_df = df.groupBy(df.columns).count().filter("count > 1")

duplicate_df.show()

+--------------------+-----------------------+--------------------+------------+-----------------------+-------------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+-------------+--------------+------+--------+-----+
|             company|company_public_response|    company_response|complaint_id|complaint_what_happened|consumer_consent_provided|consumer_disputed|       date_received|date_sent_to_company|               issue|             product|state|           sub_issue|         sub_product|submitted_via|          tags|timely|zip_code|count|
+--------------------+-----------------------+--------------------+------------+-----------------------+-------------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+-------------+--------------+------+--------+-----+
|Arm

In [21]:
df= df.dropDuplicates()

In [22]:

def perform_null_analysis(df,table_name):
    null_value_analysis=[]
    n_row=df.count()
    for column in df.columns:
        print(column)
        response = sql(f"select {n_row} as  total_row,count(*) as null_row_{column},(count(*)*100)/{n_row} as missing_percentage,'{column}' as  column_name from {table_name} where {column} is null").collect()
        null_value_analysis.append(response)
    return null_value_analysis

In [23]:
null_report = perform_null_analysis(df,complaint_table)

company
company_public_response
company_response
complaint_id
complaint_what_happened
consumer_consent_provided
consumer_disputed
date_received
date_sent_to_company
issue
product
state


sub_issue


sub_product


submitted_via
tags
timely
zip_code


In [24]:
def unwanted_column_by_missing_percentage(null_value_analysis,per_thres=20):
    columns= []
    for row in null_value_analysis:
        row_info=row[0]
        if row_info.missing_percentage>per_thres:
            print(row_info)
            columns.append(row_info.column_name)

    return columns


In [ ]:
columns = unwanted_column_by_missing_percentage(null_value_analysis=null_report)

Row(total_row=236431, null_row_company_public_response=217056, missing_percentage=91.80522012764824, column_name='company_public_response')
Row(total_row=236431, null_row_sub_issue=213539, missing_percentage=90.31768253739992, column_name='sub_issue')
Row(total_row=236431, null_row_sub_product=153327, missing_percentage=64.85063295422343, column_name='sub_product')
Row(total_row=236431, null_row_tags=379289, missing_percentage=160.42270260668016, column_name='tags')


26/05/25 07:00:12 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 187036 ms exceeds timeout 120000 ms
26/05/25 07:00:12 WARN SparkContext: Killing executors is not supported by current scheduler.


In [ ]:
def drop_column(df,columns):
    selected_column = list(filter(lambda x:x not in columns,df.columns))
    selected_column = ",".join(selected_column)
    df= sql(f"select {selected_column}  from {complaint_table} ")
    return df

In [ ]:
df=drop_column(df,columns)

In [ ]:
columns = perform_null_analysis(df,complaint_table)

In [ ]:
unwanted_column_by_missing_percentage(columns)

In [ ]:
#dropping feature as we have found more than 20% of null value in above columns

In [ ]:
df.collect()

In [ ]:
## Unique values in each columns



In [ ]:
print(f"Total number of row: {df.count()}")
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

In [ ]:
update_column_attribute(df)

In [ ]:
df=drop_column(df,columns=[df.complaint_id])

In [ ]:
update_column_attribute(df)

In [ ]:
df.printSchema()

In [ ]:
print(f"Total number of row: {df.count()}")
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

In [ ]:
df.groupBy(df.company_response).count().show()
df.groupBy(df.consumer_consent_provided).count().show()
df.groupBy(df.consumer_disputed).count().show()
df.groupBy(df.product).count().show()
df.groupBy(df.submitted_via).count().show()
df.groupBy(df.timely).count().show()


In [ ]:

df.groupBy(df.company_response).count().collect()
df.groupBy(df.consumer_consent_provided).count().collect()
df.groupBy(df.consumer_disputed).count().collect()
df.groupBy(df.product).count().collect()
df.groupBy(df.submitted_via).count().collect()
df.groupBy(df.timely).count().collect()


In [ ]:
df.groupBy(df.product).count().collect()

In [ ]:
ONE_HOT_FEATURE = [df.company_response,df.consumer_consent_provided,df.submitted_via,df.timely]
BINARY_ENCODING = [df.product,]
TARGET_FEATURE = [df.consumer_disputed]

#df.company_response No null value
#df.consumer_consent_provided  replace null with top category 
#df.consumer_disputed target feature label encoding
#df.product one hot encoding 

#df.product no null value


In [ ]:
remaining_column = list(filter(lambda x: x not in ONE_HOT_FEATURE+BINARY_ENCODING+TARGET_FEATURE,df.columns))

In [ ]:
remaining_column

In [ ]:
df.groupBy(df.company).count().count()

In [ ]:
FREQUENCY_ENCODING = [df.company]

In [ ]:
df.select(df.company).count()-df.select(df.company).dropna().count()


In [ ]:
REPLACE_NULL_WITH_TOP_VALUE = [df.zip_code,df.state,df.consumer_consent_provided]

In [ ]:
REPLACE_NULL_WITH_TOP_VALUE

In [ ]:
print(f"Total number of row: {df.count()}")
for column in remaining_column:
    print(f"{column}:  {df.select(column).distinct().count()}")

In [ ]:
df.groupBy(df.issue).count().show()

In [ ]:
remaining_column

In [ ]:
df.select(df.complaint_what_happened)[3].collect()

In [ ]:
sql(f"select {df.complaint_what_happened} from {complaint_table} limit 5").collect()

In [ ]:
TOKENIZER_FEATURE = [df.complaint_what_happened]

In [ ]:
ONE_HOT_FEATURE

In [ ]:
FREQUENCY_ENCODING = [df.company,df.issue,df.state,df.zip_code]

In [ ]:
TARGET_FEATURE

In [ ]:
TOKENIZER_FEATURE

In [ ]:
remaining_column

In [ ]:
from pyspark.sql.types import TimestampType

In [ ]:
df=df.withColumn(df.date_received,col(df.date_received).cast(TimestampType()))

In [ ]:
update_column_attribute(df)

In [ ]:
df=df.withColumn(df.date_sent_to_company,col(df.date_sent_to_company).cast(TimestampType()))
df.printSchema()

In [ ]:
df.select([df.date_received,df.date_sent_to_company]).show()

In [ ]:
from pyspark.sql.types import  LongType

In [ ]:
df = df.withColumn("diff_in_days",(col(df.date_sent_to_company).cast(LongType())-col(df.date_received).cast(LongType()))/(60*60*24))

In [ ]:
update_column_attribute(df)

In [ ]:
remove_column = [df.date_received,df.date_sent_to_company]

In [ ]:
df=df.drop(col(df.date_received)).drop(col(df.date_sent_to_company))

In [ ]:
NUMERICAL_FEATURE = [df.diff_in_days,]
ONE_HOT_FEATURE+\
FREQUENCY_ENCODING+\
BINARY_ENCODING+\
TARGET_FEATURE

In [ ]:
BINARY_ENCODING

In [ ]:
FREQUENCY_ENCODING=FREQUENCY_ENCODING+BINARY_ENCODING

In [ ]:
FREQUENCY_ENCODING.remove('issue')

In [ ]:
FREQUENCY_ENCODING

In [ ]:
ONE_HOT_FEATURE

In [ ]:

data_file_path="/home/jovyan/work/finance_complaint/finance_artifact/data_preprocessing/20220907_063829/complaint_data"

In [ ]:
from finance_complaint.entity.spark_manager import spark_session
df = spark_session.read.parquet(data_file_path)